# 🎬 Sentiment Analysis — Exploratory Data Analysis (EDA)

Dataset: IMDB Movie Reviews (50,000 reviews)  
Goal: understand the data before training the model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Loading the data

In [ ]:
df = pd.read_csv('../data/IMDB_Dataset.csv')
print(df.shape)
df.head()

In [ ]:
# Class distribution
df['sentiment'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Sentiment distribution')
plt.xticks(rotation=0)
plt.ylabel('Number of reviews')
plt.show()
print(df['sentiment'].value_counts())

## 2. Review length analysis

In [ ]:
df['word_count'] = df['review'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['review'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment, color in [('positive', '#2ecc71'), ('negative', '#e74c3c')]:
    subset = df[df['sentiment'] == sentiment]['word_count']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=sentiment)

axes[0].set_title('Word count distribution')
axes[0].set_xlabel('Number of words')
axes[0].legend()

df.boxplot(column='word_count', by='sentiment', ax=axes[1],
           boxprops=dict(color='steelblue'))
axes[1].set_title('Length by sentiment')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print(df.groupby('sentiment')['word_count'].describe())

## 3. WordCloud — most frequent words

In [ ]:
def clean_for_wc(text):
    text = re.sub(r'<.*?>', ' ', str(text))
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return text.lower()

stopwords_custom = {'the', 'a', 'an', 'and', 'is', 'it', 'in', 'of', 'to',
                    'this', 'that', 'was', 'for', 'i', 'with', 'but', 'as',
                    'on', 'are', 'be', 'have', 'film', 'movie'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, sentiment, cmap, title in zip(
    axes,
    ['positive', 'negative'],
    ['Greens', 'Reds'],
    ['Positive Reviews', 'Negative Reviews']
):
    corpus = ' '.join(
        df[df['sentiment'] == sentiment]['review'].apply(clean_for_wc)
    )
    wc = WordCloud(
        width=800, height=400, background_color='white',
        colormap=cmap, stopwords=stopwords_custom,
        max_words=100
    ).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14)

plt.tight_layout()
plt.show()

## 4. Most frequent words (top 20)

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def top_words(df, sentiment, n=20):
    corpus = ' '.join(df[df['sentiment'] == sentiment]['review'].apply(clean_for_wc))
    words = [w for w in corpus.split() if w not in ENGLISH_STOP_WORDS and len(w) > 2]
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, sentiment, color in zip(axes, ['positive', 'negative'], ['#2ecc71', '#e74c3c']):
    words, counts = zip(*top_words(df, sentiment))
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.85)
    ax.set_title(f'Top 20 words — {sentiment}')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.show()

## 5. Sample reviews

In [ ]:
print('=== POSITIVE REVIEWS (examples) ===\n')
for review in df[df['sentiment'] == 'positive']['review'].sample(2, random_state=42):
    print(review[:300], '...\n')

print('=== NEGATIVE REVIEWS (examples) ===')
for review in df[df['sentiment'] == 'negative']['review'].sample(2, random_state=42):
    print(review[:300], '...\n')

---
**Next step:** `src/train.py` to train and compare models.